E-Commerce US dataset

In [1]:
import pandas as pd
import numpy as np

In [ ]:
SAMPLE_PATH = r"C:\NG\E-Commerce US dataset\E-Commerce-US-dataset\data\raw_sample\\"
table_names = ["customers","orders","order_items","order_payments",
               "order_reviews","products","sellers","geolocation"]

tables = {}
for name in table_names:
    tables[name] = pd.read_csv(SAMPLE_PATH + f"{name}.csv")

print("All 8 tables loaded for quality assessment ")
for name, df in tables.items():
    print(f"  {name:18s}: {df.shape}")

All 8 tables loaded for quality assessment 
  customers         : (10000, 9)
  orders            : (10000, 8)
  order_items       : (21838, 8)
  order_payments    : (11474, 5)
  order_reviews     : (9332, 7)
  products          : (1984, 10)
  sellers           : (500, 8)
  geolocation       : (11255, 5)


In [ ]:
def missing_value_report(df, table_name):
    total = len(df)
    miss = df.isna().sum()                   
    miss_pct = (miss / total * 100).round(2) 

    report = pd.DataFrame({
        "table": table_name,
        "column": df.columns,
        "missing_count": miss.values,
        "missing_pct": miss_pct.values
    })
    report = report[report["missing_count"] > 0].reset_index(drop=True)
    return report

all_missing = []
for name, df in tables.items():
    r = missing_value_report(df, name)
    if len(r) > 0:
        all_missing.append(r)

missing_summary = pd.concat(all_missing, ignore_index=True) if all_missing else pd.DataFrame()
print("===== MISSING VALUE SUMMARY =====")
missing_summary

===== MISSING VALUE SUMMARY =====


,table,column,missing_count,missing_pct
0,orders,order_delivered_carrier_date,668,6.68
1,orders,order_delivered_customer_date,668,6.68


In [11]:
primary_keys = {
    "customers": "customer_id",
    "orders": "order_id",
    "order_payments": None,        
    "order_reviews": "review_id",
    "products": "product_id",
    "sellers": "seller_id",
    "order_items": None,           
    "geolocation": None
}

print("===== DUPLICATE ANALYSIS =====\n")
for name, df in tables.items():
    full_dup = df.duplicated().sum()
    print(f"{name:18s}: {full_dup} full-row duplicates", end="")

    key = primary_keys.get(name)
    if key:
        key_dup = df.duplicated(subset=[key]).sum()  
        print(f"  |  {key} duplicates: {key_dup}")
    else:
        print()

===== DUPLICATE ANALYSIS =====

customers         : 0 full-row duplicates  |  customer_id duplicates: 0
orders            : 0 full-row duplicates  |  order_id duplicates: 0
order_items       : 0 full-row duplicates
order_payments    : 0 full-row duplicates
order_reviews     : 0 full-row duplicates  |  review_id duplicates: 0
products          : 0 full-row duplicates  |  product_id duplicates: 0
sellers           : 0 full-row duplicates  |  seller_id duplicates: 0
geolocation       : 0 full-row duplicates


In [12]:
print("===== INVALID VALUE CHECKS =====\n")

issues = []

n = (tables["order_items"]["price"] <= 0).sum()
issues.append(("order_items", "price <= 0", n))

n = (tables["order_items"]["freight_value"] < 0).sum()
issues.append(("order_items", "freight_value < 0", n))

n = (tables["order_payments"]["payment_value"] <= 0).sum()
issues.append(("order_payments", "payment_value <= 0", n))

n = (tables["products"]["price"] < tables["products"]["cost"]).sum()
issues.append(("products", "price < cost (selling at loss)", n))

age = tables["customers"]["customer_age"]
n = ((age < 0) | (age > 120)).sum()
issues.append(("customers", "age out of [0,120]", n))

invalid_report = pd.DataFrame(issues, columns=["table","rule","violations"])
invalid_report

===== INVALID VALUE CHECKS =====



,table,rule,violations
0,order_items,price <= 0,0
1,order_items,freight_value < 0,0
2,order_payments,payment_value <= 0,0
3,products,price < cost (selling at loss),0
4,customers,"age out of [0,120]",0


In [13]:
print("===== CONSISTENCY CHECKS =====\n")

print("order_status values:")
print(tables["orders"]["order_status"].value_counts(), "\n")

print("payment_type values:")
print(tables["order_payments"]["payment_type"].value_counts(), "\n")

o = tables["orders"].copy()
o["pur"] = pd.to_datetime(o["order_purchase_timestamp"], errors="coerce")
o["del"] = pd.to_datetime(o["order_delivered_customer_date"], errors="coerce")
bad_dates = (o["del"] < o["pur"]).sum()
print(f"Orders delivered BEFORE purchase (impossible): {bad_dates}")

orphan_items = (~tables["order_items"]["order_id"].isin(tables["orders"]["order_id"])).sum()
print(f"Orphan order_items (no parent order): {orphan_items}")

===== CONSISTENCY CHECKS =====

order_status values:
order_status
delivered    9332
canceled      668
Name: count, dtype: int64 

payment_type values:
payment_type
credit_card      5367
paypal           1824
voucher          1242
bank_transfer     985
debit_card        938
apple_pay         865
boleto            253
Name: count, dtype: int64 

Orders delivered BEFORE purchase (impossible): 0
Orphan order_items (no parent order): 0
